In [1]:
!pip install transformers torch --quiet

print("Libraries installed successfully.")


Libraries installed successfully.


In [2]:

# PyTorch — deep learning backend
import torch
# Hugging Face model loaders
from transformers import AutoModelForCausalLM, AutoTokenizer

# for version display
import transformers

print("Imports successful.")
print(f"   PyTorch version      : {torch.__version__}")
print(f"   Transformers version : {transformers.__version__}")

Imports successful.
   PyTorch version      : 2.10.0+cpu
   Transformers version : 5.0.0


In [3]:
#   AutoTokenizer        → converts text <-> token IDs
#   AutoModelForCausalLM → transformer that predicts the next token

MODEL_NAME = "microsoft/DialoGPT-medium"

# Load tokenizer — maps words/sub-words to integer token IDs
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the causal language model weights from Hugging Face Hub
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode (disables dropout layers)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print("Model and tokenizer loaded successfully!")
print(f"Total parameters: {total_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and tokenizer loaded successfully!
Total parameters: 406,286,336


In [4]:
def generate_response(user_input: str, chat_history_ids=None, max_length: int = 1000):

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    )

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=max_length,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.75,
            repetition_penalty=1.3,
        )

    response_tokens = chat_history_ids[:, bot_input_ids.shape[-1]:]
    response_text = tokenizer.decode(
        response_tokens[0],
        skip_special_tokens=True
    )

    return response_text, chat_history_ids


print("generate_response() function defined successfully.")

generate_response() function defined successfully.


In [5]:

# Verify the model is working before running the full chatbot

test_input = "Hello, how are you?"
response, _ = generate_response(test_input)

print(f"User   : {test_input}")
print(f"Chatbot: {response}")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


User   : Hello, how are you?
Chatbot: I'm doing well, thanks! How about yourself?


In [ ]:
def run_chatbot():
    print("=" * 60)
    print("  DialoGPT Chatbot  |  Hugging Face Transformers")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 60)
    print("  Type  exit  or  quit  to end the conversation.")
    print("-" * 60)

    chat_history_ids = None

    while True:
        # accept user input
        user_input = input("You: ").strip()

        # Skip blank / whitespace-only input
        if not user_input:
            print("Chatbot: Please say something. I'm listening!")
            continue

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! It was wonderful talking with you. Have a great day!👋")
            print("=" * 60)
            break

        # Generate response using the transformer model
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        # Guard against rare empty-response edge case
        if not response.strip():
            response = "I am not quite sure how to respond to that. Could you rephrase?"

        print(f"Chatbot: {response}")
        print()
run_chatbot()
